# Usage demo

This notebook walks through the practical local-usage pattern for the
paper's three DQC-aware SABRE variants and the QIG partitioning
pre-processing step, then separately demonstrates pytket-dqc's
superconducting-topology-aware distribution. See
[`MODIFICATIONS.md`](../MODIFICATIONS.md) for what each piece does and how
it maps to the paper.

This is a simplified, from-scratch version of the pattern used in the
paper's own (SLURM-adapted) experiment scripts: a real multi-QPU backend,
an `AnalysisPass` that injects a fixed starting layout, and one call per
SABRE variant. Simplified here means: no real IBM backend / SLURM /
multiprocessing / experiment-database bookkeeping -- just the algorithmic
pipeline itself, on a small toy example anyone can run.

**Verification status** (see the README's "Reproducing the environment"
and "Using QIG partitioning" sections for details):

- Sections 1-2 (QIG partitioning, all three SABRE variants) have been run
  end-to-end against a real build of the patched Qiskit and the
  `qig-partitioning` package, and produce the claimed effect: (1,10)
  SABRE and CLA-SABRE both route with fewer inter-QPU SWAP crossings than
  Default SABRE on the same QIG-derived initial layout.
- Section 3 (pytket-dqc) has been checked for API correctness (imports,
  gateset requirements, `server_link_capacities`) but **not run
  end-to-end**: distribution requires KaHyPar, and the only version pip
  can install (1.3.7) is incompatible with this pytket-dqc version's
  partitioning code (fails even on pytket-dqc's own unmodified test
  suite). Getting a working KaHyPar (1.3.2, built from source) is a
  separate, environment-specific task — see the README.


## 1. Setup: a small multi-QPU coupling map, a circuit, and QIG partitioning

Three toy "QPUs" (cores) of 4 qubits each, wired in a **line topology**
(core 0 &harr; core 1 &harr; core 2 -- no direct core-0-to-core-2 link, so
some inter-core communication has to hop through core 1). Each core is
internally all-to-all coupled, for simplicity.

The circuit has three tightly-coupled qubit clusters (one per intended
core) plus a few deliberately "expensive" cross-cluster gates -- including
one between the two cores that *aren't* directly linked, forcing a hop.

QIG partitioning (`qig_partitioning.get_heterogeneous_core_assignment`,
paper §V-C) then assigns each virtual qubit to a physical core, matching
abstract KaHyPar blocks to real cores by communication cost and enforcing
the per-core capacity limit -- entirely independent of either patched
fork, hence its own package rather than living in the Qiskit patch.


In [ ]:
import random
import networkx as nx
from qiskit import QuantumCircuit
from qiskit.transpiler import CouplingMap, Layout, PassManager, AnalysisPass
from qiskit.transpiler.passes import SabreLayout
from qig_partitioning import get_heterogeneous_core_assignment

N_PER_CORE = 4
core_ranges = {0: range(0, 4), 1: range(4, 8), 2: range(8, 12)}

# All-to-all within each core...
local_edges = [
    (u, v)
    for qs in core_ranges.values()
    for u in qs for v in qs if u != v
]
# ...line topology between cores: only core0<->core1 and core1<->core2 are linked.
cross_edges = [(3, 4), (4, 3), (7, 8), (8, 7)]
coupling_map = CouplingMap(local_edges + cross_edges)

qubit_qpu_map = [q // N_PER_CORE for q in range(12)]
inter_qpu_coupling_map = [(0, 1), (1, 0), (1, 2), (2, 1)]  # core-level line topology

qc = QuantumCircuit(12)
for qs in core_ranges.values():
    qs = list(qs)
    for _ in range(15):
        for i in range(len(qs) - 1):
            qc.cx(qs[i], qs[i + 1])
qc.cx(1, 5)    # core 0 <-> core 1 (direct link)
qc.cx(5, 9)    # core 1 <-> core 2 (direct link)
qc.cx(2, 10)   # core 0 <-> core 2 (no direct link -- must hop through core 1)

# Communication cost between cores, matching the line topology (no direct 0<->2 link).
qig_cost_matrix = {
    0: {0: 0.0, 1: 1.0, 2: 2.0},
    1: {0: 1.0, 1: 0.0, 2: 1.0},
    2: {0: 2.0, 1: 1.0, 2: 0.0},
}
assignment = get_heterogeneous_core_assignment(qc, qig_cost_matrix, max_capacity=N_PER_CORE)

by_core = {}
for q, core in assignment.items():
    by_core.setdefault(core, []).append(qc.find_bit(q).index)
for core in sorted(by_core):
    print(f"core {core}: qubits {sorted(by_core[core])}")


## 2. Injecting the QIG layout into the three DQC-aware SABRE variants

QIG partitioning only decides *which core* each virtual qubit goes to,
not which physical qubit within that core -- so we randomly assign
physical qubits within each core, then build a Qiskit `Layout` from it.

`InjectStartingLayout` is a five-line `AnalysisPass` that writes to
`property_set["sabre_starting_layouts"]` -- a mechanism that already
exists in upstream Qiskit (not something this repo's patch adds).
Combined with `layout_trials=0` (which, via the patch's
`num_random_trials > 1` gate described in `MODIFICATIONS.md`, skips
SABRE's usual random-layout search), this forces every variant below to
route starting from exactly the QIG-derived layout, so the comparison
between variants isn't confounded by different starting points.


In [ ]:
random.seed(0)
layout_map = {}
for core, qubit_indices in by_core.items():
    available = list(core_ranges[core])
    for q_idx in sorted(qubit_indices):
        phys = random.choice(available)
        available.remove(phys)
        layout_map[qc.qubits[q_idx]] = phys
initial_layout = Layout(layout_map)


class InjectStartingLayout(AnalysisPass):
    def __init__(self, layouts):
        super().__init__()
        self.custom_layouts = layouts

    def run(self, dag):
        self.property_set["sabre_starting_layouts"] = self.custom_layouts


def generate_custom_distance_matrix(factor):
    """(1,10) SABRE's namesake weighting: local edges cost `factor`, inter-core edges cost 1."""
    weighted_edges = [
        (a, b, 1) if (a, b) in cross_edges else (a, b, factor)
        for a, b in coupling_map.get_edges()
    ]
    G = nx.DiGraph()
    G.add_weighted_edges_from(weighted_edges)
    return nx.floyd_warshall_numpy(G, range(coupling_map.size()))


def run_variant(name, **kwargs):
    pm = PassManager([
        InjectStartingLayout([initial_layout]),
        SabreLayout(
            coupling_map, routing_pass=None, seed=0, layout_trials=0,
            max_iterations=3, swap_trials=5, **kwargs,
        ),
    ])
    routed = pm.run(qc)
    cross_edges_set = {tuple(sorted(e)) for e in cross_edges}
    inter_swaps = sum(
        1 for instr in routed.data
        if instr.operation.name == "swap"
        and tuple(sorted((routed.find_bit(instr.qubits[0]).index, routed.find_bit(instr.qubits[1]).index))) in cross_edges_set
    )
    total_swaps = sum(1 for instr in routed.data if instr.operation.name == "swap")
    print(f"{name}: total swaps = {total_swaps}, inter-QPU swaps = {inter_swaps}")
    return routed


_ = run_variant("Default SABRE")
_ = run_variant("(1,10) SABRE", custom_distance_matrix=generate_custom_distance_matrix(10).tolist())
_ = run_variant(
    "CLA-SABRE",
    penalized_swaps=cross_edges,
    qubit_qpu_map=qubit_qpu_map,
    inter_qpu_coupling_map=inter_qpu_coupling_map,
    # alpha / beta left at their defaults (9.0, 3.0) -- see MODIFICATIONS.md's "CLA-SABRE" section.
)


## 3. Superconducting topology-aware distribution (pytket-dqc)

A different paradigm from the Qiskit-side SABRE variants above: rather
than routing SWAPs, pytket-dqc distributes a circuit across cores using
entanglement (ebits) for non-local gates. A `NISQNetwork` with
`server_link_capacities` set per-edge (rather than relying on the
uniform, per-server `server_ebit_mem`) is this repo's adaptation for
superconducting hardware -- see `MODIFICATIONS.md`'s "pytket-dqc:
superconducting hardware adaptation" section for why that's the
meaningful change.


In [ ]:
from pytket import Circuit, OpType
from pytket_dqc.networks import NISQNetwork
from pytket_dqc.allocators import HypergraphPartitioning
from pytket_dqc.utils import DQCPass

# Two 2-qubit servers connected by a single link with capacity 2.
network = NISQNetwork(
    server_coupling=[[0, 1]],
    server_qubits={0: [0, 1], 1: [2, 3]},
    server_link_capacities={(0, 1): 2},
)

circ = (
    Circuit(4)
    .add_gate(OpType.CU1, 1.0, [0, 2])
    .H(0)
    .add_gate(OpType.CU1, 1.0, [1, 3])
    .H(1)
    .add_gate(OpType.CU1, 1.0, [0, 2])
    .add_gate(OpType.CU1, 1.0, [1, 3])
)
DQCPass().apply(circ)  # rebases to pytket-dqc's required gateset

distribution = HypergraphPartitioning().allocate(circ, network, seed=0)
print("ebit cost:", distribution.cost())

distributed_circ = distribution.to_pytket_circuit()
print(distributed_circ)
